In [ ]:
def main(datasources, start_date, end_date):
    """构造一个确定性的伪随机差因子，预期 IC 和多空收益接近 0。"""
    import dai
    import numpy as np
    import pandas as pd

    pool = dai.query(
        "SELECT date, instrument FROM cpt_jyc_2026_instruments",
        filters={'date': [start_date, end_date]},
        # 不启用字典压缩，避免评估模块将 Arrow uint16 字典索引
        # 转回 pandas 时触发 ArrowTypeError。
        compression=False,
    ).df()
    pool['date'] = pd.to_datetime(pool['date'])
    pool['instrument'] = pool['instrument'].astype(str)

    # 只依赖股票代码和时间戳，不使用任何行情或未来数据。
    code = pd.to_numeric(pool['instrument'].str.slice(0, 6), errors='coerce').fillna(0)
    minute_key = (
        pool['date'].dt.year.astype('int64') * 600_000
        + pool['date'].dt.dayofyear.astype('int64') * 1_440
        + pool['date'].dt.hour.astype('int64') * 60
        + pool['date'].dt.minute.astype('int64')
    )
    seed = (code.astype('int64') * 1_000_003 + minute_key * 97) % 2_147_483_647
    pool['factor'] = np.sin(seed * 12.9898).astype('float64')
    return pool[['date', 'instrument', 'factor']].reset_index(drop=True)


if __name__ == '__main__':
    from bigmodule import M

    datasources = {'bar1m': 'cpt_jyc_2026_stock_bar1m'}
    start_date = '2025-01-01 00:00:00'
    end_date = '2025-12-31 23:59:59'
    factor_data = main(datasources, start_date, end_date)
    result = M.jyc_eval._latest(factor_data=factor_data, show=True)
